# 🧠 Batch Size Analysis: Noise, Memory, and Scaling Rules

Welcome to the hands-on explanation notebook for **Batch Size Analysis**! In this notebook, we will:
1. Formulate the relationship between batch size and gradient noise (Central Limit Theorem).
2. Generate synthetic data and numerically compute the variance of gradient estimates across different batch sizes.
3. Plot the **Gradient Noise vs. Batch Size** curve to visualize the $1/\sqrt{B}$ noise decay law.
4. Implement the **Linear Scaling Rule** for adjusting learning rates when scaling batch sizes.
5. Estimate VRAM memory scaling constraints.
6. Connect these principles to YOLO's batch configuration and CUDA Out of Memory (OOM) troubleshooting.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Quantifying Gradient Noise Numerically

According to the Central Limit Theorem, the variance of the sample mean gradient decreases in proportion to $1/B$ (where $B$ is the batch size). Thus, the standard deviation (the noise) decays as:
$$\text{Noise} \propto \frac{1}{\sqrt{B}}$$

Let's generate 200 data points and evaluate this relationship.

In [ ]:
n_samples = 200
X = np.random.rand(n_samples, 1)
y = 2.0 * X + 1.0 + np.random.normal(0, 0.2, (n_samples, 1))

# We evaluate the gradient at a fixed weight w=1.5 and bias b=0.5
w_test, b_test = 1.5, 0.5

def single_sample_gradient(xi, yi, w, b):
    pred = w * xi + b
    return (pred - yi) * xi

true_grads = [single_sample_gradient(X[i], y[i], w_test, b_test) for i in range(n_samples)]
true_mean_grad = np.mean(true_grads)

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
std_deviations = []

for B in batch_sizes:
    batch_gradients = []
    for _ in range(150):
        indices = np.random.choice(n_samples, size=B, replace=False)
        sample_grads = [single_sample_gradient(X[i], y[i], w_test, b_test) for i in indices]
        batch_gradients.append(np.mean(sample_grads))
    
    std_deviations.append(np.std(batch_gradients))

Let's plot the standard deviation of gradient estimates vs. batch size, overlaying the theoretical $1/\sqrt{B}$ decay curve.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(batch_sizes, std_deviations, color='red', marker='o', linewidth=2.5, label='Empirical Gradient Noise (Std Dev)')

theoretical = std_deviations[0] / np.sqrt(batch_sizes)
plt.plot(batch_sizes, theoretical, color='blue', linestyle='--', linewidth=2, label='Theoretical 1/√B Decay')

plt.xscale('log', base=2)
plt.xlabel('Batch Size (B)')
plt.ylabel('Gradient Estimate Noise (Std Dev)')
plt.title('Gradient Noise Reduction vs. Batch Size')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.legend()
plt.show()

Look at the plot!
-   **Batch Size = 1 (Stochastic GD):** Extreme noise. The gradient updates can point in wildly wrong directions on individual steps, but they introduce helpful regularization.
-   **Batch Size = 32 or 64:** Noise is compressed by a factor of 5-8. The steps are highly stable and reliable.
-   The empirical data matches the theoretical $1/\sqrt{B}$ decay curve almost perfectly!

## 2. The Linear Scaling Rule

When scaling batch size from a base configuration, we scale the learning rate proportionally:
$$\text{lr}_{\text{scaled}} = \text{lr}_{\text{base}} \times \frac{\text{batch}_{\text{target}}}{\text{batch}_{\text{base}}}$$

In [ ]:
def scale_learning_rate(lr_base, batch_base, batch_target):
    scale_factor = batch_target / batch_base
    return lr_base * scale_factor

print("Scaled LR at batch=64:", scale_learning_rate(0.01, 16, 64))

## 💡 Connection to YOLO and CUDA OOM
*   **VRAM footprint:** Inside your YOLO configuration logs, you can see batch size settings (e.g. `batch=16`). Memory scales linearly with batch size, but **quadratically** with image size:
    $$\text{VRAM} \propto \text{Batch} \times \text{ImageHeight} \times \text{ImageWidth}$$
*   **CUDA OOM:** If your GPU throws a CUDA Out of Memory error:
    1.  Divide `batch` by 2 (e.g. 16 to 8).
    2.  If that is not enough, reduce `imgsz` (e.g. 640 to 480).
    3.  If you want to maintain the effective batch size of 16 but only have VRAM for batch size 8, update gradients every 2 steps (gradient accumulation).